In [3]:
from __future__ import annotations

import hashlib
import json
from pathlib import Path
from typing import Iterator

import polars as pl

from sec_edgar.client import EdgarClient
import edgar as edg
import sec2md

In [ ]:
tra_variants = ['"tax receivable agreement"','"tax receivable agreements"', 
'"tax receivables agreement"', '"tax receivables agreements"'
]

tra_variants = '"tax receivable agreement" OR "tax receivable agreements" OR "tax receivables agreement" OR "tax receivables agreements"'
tra_lone = '"TRA"' #Note the quotation marks inside the string


edg.set_identity("sulli98@uw.edu")
results = (
    edg.search_filings(
        '',
        forms=['8-K', '10-K']
)


In [5]:
results

EFTSSearch(query='"tax receivable agreement" OR "tax receivable agreements" OR "tax receivables agreement" OR "tax receivables agreements"', total=9799, showing=20)

In [6]:
r = results[0]
r

EFTSResult(26.0 | 10-K | EX-10.14 | Norcraft Companies, Inc.  (CIK 0001582616) | 2014-03-31)

In [ ]:
# SEC-accepted format: "<Organization or Name> <contact email>".
# The contact email is project-fixed; see ca-PLAN.md.
USER_AGENT = "tra-research-pipeline Alex Sullivan sulli98@uw.edu"

# Conservative target; the SEC cap is 10/sec and a small safety margin
# absorbs network and timing jitter.
DEFAULT_RATE_PER_SEC = 9.0

CACHE_ROOT = Path(".tra_history_cache/edgar_search")
EDGAR_FULLTEXT_URL = "https://efts.sec.gov/LATEST/search-index"
DEFAULT_MAX_AGE_S = 24 * 3600  # 1 day; search index is live
PAGE_CAP = 10000  # SEC: from + size <= 10000

# SEC documented maximum is 100, but the FTS backend returns 500
# Internal Server Error deterministically for short-phrase queries
# (e.g. ``q='"TRA"'``) at certain offset/size combinations around 200-299
# with size=100. Halving to 50 dodges the bug at the cost of an extra
# paginated request per ~100 results. Verified 2026-05-26 against
# ``q='"TRA"'`` for 2024-02 where size=100 at offset=200 returns 500
# and size=50 at the same offset returns 200.
PAGE_SIZE = 50